# Governance & Security with Unity Catalog

Access control, data masking, row-level security, and observability with Unity Catalog.

> **Exam domain (exam guide updated August 6, 2026):** Governance and Security — 15%

| Domain | Key Concept | Exam Keywords |
|--------|-------------|---------------|
| Access Control | GRANT / REVOKE (DENY = legacy Hive metastore only) | privilege hierarchy, principal |
| Data Masking | Column mask functions | `SET MASK`, `DROP MASK` |
| Row Security | Row filters | `SET ROW FILTER`, `DROP ROW FILTER` |
| Table Lifecycle | Managed vs External (module 03) | `DROP TABLE` behavior, `ALTER TABLE … SET MANAGED` |
| Lineage *(beyond exam)* | System tables | `system.access.audit`, `system.access.table_lineage` |


## Setup

> **After a break:** the serverless session can expire. Later cells then fail with `NameError: name 'CATALOG' is not defined`. Open the cell menu of the cell where you stopped and select **Run all above**. It runs Setup again and reloads `bronze.customers`.

> **Trainer:** do not take a break between Step 2 (attach the mask and the filter) and Step 4 (cleanup). Until Step 4 runs, the mask and the filter stay on `bronze.customers`.

In [0]:
%run ../../setup/00_setup

### Configuration
Define the path to the customers file used throughout this notebook.

In [0]:
# Path to the customers file (a subdirectory of DATASET_PATH from 00_setup)
CUSTOMERS_PATH = f"{DATASET_PATH}/customers"
CUSTOMERS_CSV = f"{CUSTOMERS_PATH}/customers.csv"

## Unity Catalog Architecture

Unity Catalog is the governance layer of the Databricks lakehouse. It controls access to data, ML assets and BI in one place.

### Object hierarchy

<img src="../../../assets/images/nb01-uc-namespace.png" alt="Unity Catalog hierarchy metastore, catalog, schema, table and volume, with a three-level name split into catalog, schema and table" width="800">

### Three-level namespace
```sql
catalog.schema.table
```

Example:
```sql
main.sales.orders
dev.analytics.customer_metrics
prod.gold.daily_revenue
```

### Key features
- Unified governance: one platform for data, ML and BI
- Fine-grained access control on table, column and row level
- Automatic lineage: end-to-end tracking of data flow
- Audit logging: who accessed what and when
- Data discovery: metadata search and tagging

## Access Management: GRANT / REVOKE

Unity Catalog controls access with SQL-standard `GRANT` and `REVOKE` commands. Use them to decide who can read or change which objects.

### Permission hierarchy

You must grant each level from the top down.

```
CATALOG → SCHEMA → TABLE / VIEW / FUNCTION
```

| Permission | Grants access to |
|---|---|
| `USE CATALOG` | Opens the catalog (required first) |
| `USE SCHEMA` | Opens a schema inside the catalog |
| `SELECT` | Read data from a table / view |
| `MODIFY` | INSERT, UPDATE, DELETE on a table |
| `CREATE TABLE` | Create new tables inside a schema |
| `EXECUTE` | Call a function / UDF |
| `ALL PRIVILEGES` | All of the above at once |

You can grant to users (`user@company.com`) or to groups. Groups scale better, so prefer them.

> **Important:** Always grant `USE CATALOG` before `USE SCHEMA`, and `USE SCHEMA` before `SELECT`.

<img src="../../../assets/images/nb10-privileges.png" alt="To read one table the group analysts needs three grants: USE CATALOG, USE SCHEMA and SELECT" width="800">

### Groups used in this demo
The grants below go to the account group `analysts`, which gets read access to Silver and Gold. Masks and filters check membership of an account group with `is_account_group_member()`.

The demo switches that group. First it uses `account users` (every user of the account), so you see the clear data. Then it uses `pii_readers`. Nobody here is in that group, so you see what a user outside the privileged group sees.

> **Trainer prerequisite:** Unity Catalog privileges can only be granted to **account-level** groups. Workspace-local groups are rejected.

In [0]:
# Check that the schemas exist
spark.sql(f"SHOW SCHEMAS IN {CATALOG}").display()

### Active catalog and schema

Set the default catalog and schema. Later statements use them unless you give a full path.

In [0]:
# Set the default catalog and schema
# (00_setup already ran USE CATALOG; statements below still use full names)
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")
display(spark.sql("SELECT current_catalog() AS catalog, current_schema() AS schema"))

## Data Preparation

Load the customers file from the dataset directory into the Bronze table `customers`. The grants, masks, filters and the ABAC demo below all use this one table.

In [0]:
# Reload customers from CSV: the masks and filters below attach to this table
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.customers")

customers_df = spark.read.option("header", "true").option("inferSchema", "true").csv(CUSTOMERS_CSV)
(customers_df.write.format("delta").mode("overwrite")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers"))

display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers"))

### Permissions for analysts

The `analysts` group gets these grants:
- `USE CATALOG` on the catalog
- `USE SCHEMA` and `SELECT` on the Silver and Gold schemas
- `SELECT` on one Bronze table (table-level grant)

To grant on a catalog or schema you need to own it or have `MANAGE` on it. The course setup (`00_pre_config`) gives each participant `MANAGE` on their own catalog.

Expected result: `SHOW GRANTS` on the Silver schema lists `analysts` with `USE SCHEMA` and `SELECT`. `SHOW GRANTS` on `bronze.customers` lists `analysts` with `SELECT` on the table. The other rows have ObjectType `CATALOG`: privileges on the whole catalog, for example your own.

### Setup: the `analysts` account group must exist
> **Note:** An account admin creates account groups in the account console (**User management** → **Groups** → **Add group**). In an identity-federated workspace, a workspace admin can create them under **Settings → Identity and access → Groups → Manage → Add Group → Add new**. `00_pre_config` Step 5 creates `analysts` for this course. If the group is missing, the GRANT cells below stop with `PRINCIPAL_DOES_NOT_EXIST`.

In [ ]:
# Catalog level: analysts may open the catalog
spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `analysts`")
print("analysts: USE CATALOG on the catalog")

In [0]:
# Schema level: analysts may open and read every table in Silver and Gold
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{SILVER_SCHEMA} TO `analysts`")
spark.sql(f"GRANT SELECT ON SCHEMA {CATALOG}.{SILVER_SCHEMA} TO `analysts`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{GOLD_SCHEMA} TO `analysts`")
spark.sql(f"GRANT SELECT ON SCHEMA {CATALOG}.{GOLD_SCHEMA} TO `analysts`")
print("analysts: USE SCHEMA and SELECT on Silver and Gold. The next cell shows the Silver grants.")

In [ ]:
# Silver schema: analysts got USE SCHEMA and SELECT (Gold got the same)
spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOG}.{SILVER_SCHEMA}").display()

In [0]:
# Table level: analysts may read bronze.customers only, not the whole Bronze schema
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{BRONZE_SCHEMA} TO `analysts`")
spark.sql(f"GRANT SELECT ON TABLE {CATALOG}.{BRONZE_SCHEMA}.customers TO `analysts`")
print("analysts: SELECT on bronze.customers. The next cell shows the table grants.")

In [ ]:
# bronze.customers: analysts got SELECT on this one table
spark.sql(f"SHOW GRANTS ON TABLE {CATALOG}.{BRONZE_SCHEMA}.customers").display()

## REVOKE and DENY

The exam guide (updated August 6, 2026) lists **GRANT / REVOKE / DENY**. The three statements behave differently on each platform.

| Statement | Unity Catalog | Legacy Hive metastore table ACLs |
|---|---|---|
| `GRANT priv ON obj TO principal` | Yes. Allows access (inherited downward: catalog → schema → table) | Yes |
| `REVOKE priv ON obj FROM principal` | Yes. Removes a grant on the object where it was granted | Yes |
| `DENY priv ON obj TO principal` | No. Not supported: the SQL `DENY` statement applies only to the `hive_metastore` catalog | Yes. DENY overrides GRANT |

In Unity Catalog, privileges are additive: no grant means no access.
- To take access away, `REVOKE` it at the level where it was granted.
- A schema-level `SELECT` cannot be removed for one table. Grant per table instead.
- To hide rows or columns, use row filters, column masks or ABAC policies.
- ABAC DENY policies (Beta) can currently deny only `MANAGE ACCESS CONTROL`, not `SELECT`.

```sql
-- Unity Catalog
REVOKE SELECT ON TABLE catalog.bronze.customers FROM `analysts`;

-- Legacy hive_metastore table ACLs only (for recognition; fails on UC objects)
DENY SELECT ON TABLE hive_metastore.default.orders TO `contractors`;
```

> **Exam tip:** In legacy table ACLs, DENY wins over GRANT. Unity Catalog has no DENY as a SQL statement for data privileges (ABAC DENY policies are in Beta). Use REVOKE and finer-grained grants instead.

In [0]:
# Take back the table-level grant from the previous step
# (USE SCHEMA on Bronze stays: harmless without SELECT, revoked in the Step 4 cleanup)
spark.sql(f"REVOKE SELECT ON TABLE {CATALOG}.{BRONZE_SCHEMA}.customers FROM `analysts`")
print("REVOKE done. The next cell shows the result.")

In [ ]:
# bronze.customers: the analysts row with SELECT is gone
spark.sql(f"SHOW GRANTS ON TABLE {CATALOG}.{BRONZE_SCHEMA}.customers").display()

> **Managed vs external tables** (objective 7.1: create, `DROP TABLE` behaviour, `SET MANAGED` conversion) are covered in module 03, section *Managed vs External Tables*.

### Native Column Masks & Row Filters (ALTER TABLE)

Before native masks, teams hid PII with dynamic views, for example `CREATE VIEW gold.customers_v AS SELECT customer_id, CASE WHEN is_account_group_member('admins') THEN email ELSE '***' END AS email FROM bronze.customers`.

Unity Catalog also supports native column masks and row filters applied directly to tables. Prefer them: the table enforces security however users query the data.

| Feature | View-Based (dynamic view) | Native (ALTER TABLE) |
|---------|------------------------|---------------------|
| Scope | Only when querying the view | Every query on the table |
| Setup | CREATE VIEW with CASE WHEN | CREATE FUNCTION + ALTER TABLE |
| Maintenance | Must update view if logic changes | Update function only |

A row filter decides which rows a user sees. A column mask decides what a user sees in one column. Unity Catalog runs both functions on every query. In the picture the privileged group is a placeholder. In the demo below it is `account users`, then `pii_readers`.

<img src="../../../assets/images/nb10-row-filter-mask.png" alt="One table with a row filter and a column mask, read by a member and a non-member of the privileged group" width="800">

Syntax reference for column masks and row filters.

| Operation | Syntax |
|-----------|--------|
| Create mask function | `CREATE FUNCTION mask_email(email STRING) RETURN CASE WHEN is_account_group_member('admins') THEN email ELSE '***' END` |
| Apply column mask | `ALTER TABLE t ALTER COLUMN email SET MASK catalog.schema.mask_email` |
| Drop column mask | `ALTER TABLE t ALTER COLUMN email DROP MASK` |
| Create row filter | `CREATE FUNCTION filter_fn(status STRING) RETURN is_account_group_member('admins') OR status = 'active'` |
| Apply row filter | `ALTER TABLE t SET ROW FILTER catalog.schema.filter_fn ON (status)` |
| Drop row filter | `ALTER TABLE t DROP ROW FILTER` |

#### Step 1: The mask and the row filter functions

Both functions ask one question: is the user in the privileged **account** group? `is_account_group_member()` checks account-level groups, and it is the function the exam expects. `is_member()` checks the groups of the current workspace. Here both functions trust `account users`. Step 3 creates them again for `pii_readers`. Each function cell ends with one call of the function, so you see what it returns for you.

In [ ]:
# Show the current user and the group memberships the functions will check
display(spark.sql("SELECT current_user() AS me, "
                  "is_account_group_member('account users') AS in_account_users, "
                  "is_account_group_member('pii_readers') AS in_pii_readers"))

In [0]:
# Mask function: members of `account users` see the e-mail, others ***@domain
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{GOLD_SCHEMA}.mask_email_native(email STRING)
    RETURNS STRING
    RETURN CASE
        WHEN is_account_group_member('account users') THEN email
        ELSE concat('***@', substring_index(email, '@', -1))
    END
""")
# You are in `account users`, so the e-mail comes back unchanged
display(spark.sql(f"SELECT {CATALOG}.{GOLD_SCHEMA}.mask_email_native('anna@example.com') AS email"))

In [0]:
# Row filter function: members of `account users` see every row, others only Premium
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{BRONZE_SCHEMA}.filter_active_customers(
        customer_segment STRING)
    RETURNS BOOLEAN
    RETURN IF(is_account_group_member('account users'), true,
              customer_segment = 'Premium')
""")
# true: a Basic customer row stays visible for you
display(spark.sql(f"SELECT {CATALOG}.{BRONZE_SCHEMA}.filter_active_customers('Basic') AS keeps_basic_row"))

#### Step 2: Attach them to the table with ALTER TABLE

`SET MASK` attaches the mask to the column. `SET ROW FILTER` attaches the filter to the table. From now on every query on the table goes through them, whoever runs it and however they query it.

In [ ]:
# Attach the mask to the email column
spark.sql(f"""
    ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.customers
    ALTER COLUMN email SET MASK {CATALOG}.{GOLD_SCHEMA}.mask_email_native
""")
print("bronze.customers.email now goes through mask_email_native")

In [0]:
# Attach the row filter to the table
spark.sql(f"""
    ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.customers
    SET ROW FILTER {CATALOG}.{BRONZE_SCHEMA}.filter_active_customers ON (customer_segment)
""")
print("bronze.customers rows now go through filter_active_customers")

In [0]:
# Query as a member of `account users`: every segment, real e-mail addresses
display(spark.sql(f"""
    SELECT customer_segment, count(*) AS customers, max(email) AS sample_email
    FROM {CATALOG}.{BRONZE_SCHEMA}.customers
    GROUP BY customer_segment
    ORDER BY customer_segment
"""))

#### Step 3: The same table for a user outside the group

Point the functions at `pii_readers`, a group nobody here belongs to. The table and its policies do not change. Only the answer of `is_account_group_member()` changes. This is what a participant outside the privileged group sees: only Premium customers, and e-mails as `***@domain`.

In [ ]:
# Mask function again, now for `pii_readers`, a group nobody here belongs to
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{GOLD_SCHEMA}.mask_email_native(email STRING)
    RETURNS STRING
    RETURN CASE
        WHEN is_account_group_member('pii_readers') THEN email
        ELSE concat('***@', substring_index(email, '@', -1))
    END
""")
# You are not in `pii_readers`, so the same call now returns ***@example.com
display(spark.sql(f"SELECT {CATALOG}.{GOLD_SCHEMA}.mask_email_native('anna@example.com') AS email"))

In [0]:
# Row filter function again, now for `pii_readers`
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{BRONZE_SCHEMA}.filter_active_customers(
        customer_segment STRING)
    RETURNS BOOLEAN
    RETURN IF(is_account_group_member('pii_readers'), true,
              customer_segment = 'Premium')
""")
# false: a Basic customer row is now hidden from you
display(spark.sql(f"SELECT {CATALOG}.{BRONZE_SCHEMA}.filter_active_customers('Basic') AS keeps_basic_row"))

In [0]:
# Same query as in Step 2, now outside `pii_readers`: only Premium rows and masked e-mails
display(spark.sql(f"""
    SELECT customer_segment, count(*) AS customers, max(email) AS sample_email
    FROM {CATALOG}.{BRONZE_SCHEMA}.customers
    GROUP BY customer_segment
    ORDER BY customer_segment
"""))

To see the policies in Catalog Explorer, open the table and go to **Overview**. In the column list, the **Column masking** column shows `fx Column mask` for `email`; a column without a mask shows an **Add column mask** button there. The row filter is in the **Row filter** panel on the right, below **Tags**.

#### Step 4: Cleanup

Use `DROP MASK` and `DROP ROW FILTER` to remove security policies from a table, and `REVOKE` every grant the demo gave to `analysts`.

In [0]:
# Remove the column mask from email
spark.sql(f"""
    ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.customers
    ALTER COLUMN email DROP MASK
""")
print("Column mask removed")

In [0]:
# Remove the row filter from the table
spark.sql(f"""
    ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.customers
    DROP ROW FILTER
""")
print("Row filter removed")

In [0]:
# Drop the mask and the row filter functions
spark.sql(f"DROP FUNCTION IF EXISTS {CATALOG}.{GOLD_SCHEMA}.mask_email_native")
spark.sql(f"DROP FUNCTION IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.filter_active_customers")
print("Functions dropped")

In [0]:
# Revoke every grant the demo gave to analysts (REVOKE of a missing grant is a no-op)
spark.sql(f"REVOKE SELECT ON SCHEMA {CATALOG}.{SILVER_SCHEMA} FROM `analysts`")
spark.sql(f"REVOKE SELECT ON SCHEMA {CATALOG}.{GOLD_SCHEMA} FROM `analysts`")
spark.sql(f"REVOKE USE SCHEMA ON SCHEMA {CATALOG}.{BRONZE_SCHEMA} FROM `analysts`")
spark.sql(f"REVOKE USE SCHEMA ON SCHEMA {CATALOG}.{SILVER_SCHEMA} FROM `analysts`")
spark.sql(f"REVOKE USE SCHEMA ON SCHEMA {CATALOG}.{GOLD_SCHEMA} FROM `analysts`")
spark.sql(f"REVOKE USE CATALOG ON CATALOG {CATALOG} FROM `analysts`")
print("analysts has no grants from this demo any more")

## Attribute-Based Access Control (ABAC)

Row filters and column masks (previous section) are attached table by table. ABAC applies the same idea to a whole catalog or schema. You define a policy once, and it covers every table whose columns carry a matching governed tag. Row filter and column mask policies are GA since April 28, 2026. ABAC DENY policies are still Beta.

The two building blocks:

| Concept | What it is |
|---|---|
| Governed tags | Account-level tags with allowed values and rules for who may assign them. Example: the system governed tag `class.email_address`, which Data Classification sets on email columns |
| Policies | Central rules (`CREATE POLICY`) that bind a mask or row-filter function to every table/column matching a tag condition, for chosen principals |

A query is evaluated in these steps:
1. A user queries a table.
2. Unity Catalog checks the policies on the table's catalog and schema.
3. For each policy, it matches the tag conditions (`has_tag('<tag>')`, `has_tag_value('<tag>', '<value>')`) against the table's columns.
4. It applies the matching masks and filters transparently.

Why use it:
- It scales: one policy replaces `ALTER TABLE ... SET MASK` on hundreds of tables.
- You tag once and protect everywhere: new tables inherit protection the moment they are tagged.
- It separates duties: data stewards manage tags, security admins manage policies.

> **Exam Tip:** ABAC with governed tags is an objective of the exam guide (updated August 6, 2026). A policy matches columns with `has_tag` / `has_tag_value` (snake_case; the camelCase form is planned for deprecation). ABAC policies and table-level row filters / column masks **can co-exist** on a table. For a given user at query time, only **one** distinct row filter per table and **one** distinct mask per column may apply. If an ABAC policy and a table-level filter/mask would apply **different functions**, access is **blocked** with an error. If both use the same function, access is allowed. Diagnose with `SHOW EFFECTIVE POLICIES`.

<img src="../../../assets/images/nb10-abac.png" alt="ABAC: a data steward tags columns, a security admin creates one policy, and Unity Catalog masks every tagged column at query time" width="800">

### ABAC demo: one policy masks every tagged email column

The demo copies `bronze.customers` to Silver as `m10_customers`, creates a mask function and attaches one column-mask policy to the Silver schema. Creating a policy needs `MANAGE` on the schema or its catalog, which the course setup gives you on your catalog. Policies run on serverless or Databricks Runtime 16.4 and above.

In [ ]:
# Demo table in Silver: a copy of three columns of bronze.customers
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SILVER_SCHEMA}.m10_customers AS
    SELECT customer_id, first_name, email FROM {CATALOG}.{BRONZE_SCHEMA}.customers
""")
print("silver.m10_customers created")

In [0]:
# Mask function the policy will call: every e-mail becomes ***@domain
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SILVER_SCHEMA}.m10_mask_email(email STRING)
    RETURNS STRING
    RETURN CONCAT('***@', SUBSTRING_INDEX(email, '@', -1))
""")
display(spark.sql(f"SELECT {CATALOG}.{SILVER_SCHEMA}.m10_mask_email('anna@example.com') AS email"))

In [0]:
# No policy yet: the e-mails in m10_customers are in clear text
display(spark.table(f"{CATALOG}.{SILVER_SCHEMA}.m10_customers").limit(5))

**Trainer only:** tag the column. `class.email_address` is a system governed tag. Assigning it needs the `ASSIGN` permission on the tag, which by default only account admins have. The policy below masks the column as soon as someone with `ASSIGN` sets the tag.

> **Participants:** skip this cell; without the privilege it stops with a permission error. No column then has the tag, so the query after the policy shows the e-mails in clear text. The trainer's screen shows them masked.

In [ ]:
# Trainer only: tag m10_customers.email with the system governed tag class.email_address
# Participants: skip this cell; without the privilege it stops with a permission error
spark.sql(f"ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.m10_customers "
          "ALTER COLUMN email SET TAGS ('class.email_address')")
print("m10_customers.email has the tag class.email_address")

Create the policy. It covers every table in Silver, applies to all account users and masks each column that has the tag. `SHOW POLICIES` lists the policies attached to the schema.

In [ ]:
# One column-mask policy on the Silver schema for every column tagged class.email_address
spark.sql(f"""
    CREATE OR REPLACE POLICY m10_mask_tagged_emails
    ON SCHEMA {CATALOG}.{SILVER_SCHEMA}
    COMMENT 'Mask every email column tagged class.email_address'
    COLUMN MASK {CATALOG}.{SILVER_SCHEMA}.m10_mask_email
    TO `account users`
    FOR TABLES
    MATCH COLUMNS has_tag('class.email_address') AS email
    ON COLUMN email
""")
display(spark.sql(f"SHOW POLICIES ON SCHEMA {CATALOG}.{SILVER_SCHEMA}"))


In [ ]:
# Details of one policy: principals, match condition, function
display(spark.sql(f"DESCRIBE POLICY m10_mask_tagged_emails ON SCHEMA {CATALOG}.{SILVER_SCHEMA}"))


In [ ]:
# The same policy in information_schema: type and match condition
# (information_schema lists it for MANAGE holders)
pol_df = spark.sql(f"""
    SELECT policy_name, policy_type, match_columns
    FROM {CATALOG}.information_schema.abac_policy_definitions
    WHERE schema_name = '{SILVER_SCHEMA}' AND policy_name = 'm10_mask_tagged_emails'
""")
display(pol_df)

In [ ]:
# Query the table: with the tag set, every email is ***@domain;
# without it (participants skipped the tag cell) emails stay in clear text
df = spark.sql(f"SELECT customer_id, first_name, email "
               f"FROM {CATALOG}.{SILVER_SCHEMA}.m10_customers LIMIT 5")
display(df)

⊕ **Lineage.** Unity Catalog records lineage without any setup. In Catalog Explorer, open `silver.m10_customers` and go to the **Lineage** tab: the upstream table is `bronze.customers`, written by the `CREATE TABLE ... AS SELECT` above. The list also shows this notebook.

The same link is a row in the system table `system.access.table_lineage`. **Trainer only:** the cell below reads it and needs `SELECT` on `system.access`, which the course setup does not give participants. New rows can take 10 minutes or more to appear. If the query returns no rows or only older runs, run it again later.

> **Participants:** skip this cell; without the privilege it stops with a permission error.

In [ ]:
# Trainer only: the lineage row of m10_customers in the system table system.access.table_lineage
# Participants: skip this cell; without the privilege it stops with a permission error
display(spark.sql(f"""
    SELECT source_table_full_name, target_table_full_name, event_time
    FROM system.access.table_lineage
    WHERE target_table_full_name = '{CATALOG}.{SILVER_SCHEMA}.m10_customers'
      AND event_date >= current_date() - 1
    ORDER BY event_time DESC
    LIMIT 5
"""))

Clean up the demo. `DROP POLICY` has no `IF EXISTS`, so the cell checks `SHOW POLICIES` first and can run again. Dropping the table also removes its tag.

In [ ]:
# Cleanup: the policy (only if it exists), then the demo table and the mask function
policies = spark.sql(f"SHOW POLICIES ON SCHEMA {CATALOG}.{SILVER_SCHEMA}")
if policies.filter("`Policy Name` = 'm10_mask_tagged_emails'").count() > 0:
    spark.sql(f"DROP POLICY m10_mask_tagged_emails ON SCHEMA {CATALOG}.{SILVER_SCHEMA}")
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SILVER_SCHEMA}.m10_customers")
spark.sql(f"DROP FUNCTION IF EXISTS {CATALOG}.{SILVER_SCHEMA}.m10_mask_email")

# m10_mask_tagged_emails is no longer in the list
display(spark.sql(f"SHOW POLICIES ON SCHEMA {CATALOG}.{SILVER_SCHEMA}"))

## Monitoring & Observability (System Tables)

> **Self-study:** skipped in class, not required by any lab.

> **Note: beyond the current exam outline, but practical.** Audit and lineage system tables (`system.access.audit`, `system.access.table_lineage`, `system.access.column_lineage`) are not an objective of their own in the exam guide updated August 6, 2026. Its sample question 3 asks how audit logs are delivered. You will use these tables in real projects.

Unity Catalog provides system tables (`system.*`) for operational monitoring. They show costs, job runs, pipeline health, query performance and storage usage.

Available system table categories:

| Category | Schema | Key Tables |
|----------|--------|------------|
| Billing | `system.billing` | `usage`, `list_prices` |
| Compute | `system.compute` | `clusters`, `warehouse_events` |
| Lakeflow Jobs | `system.lakeflow` | `job_run_timeline`, `job_task_run_timeline` |
| Pipelines | `system.lakeflow` | `pipelines`, `pipeline_update_timeline` |
| Queries | `system.query` | `history` |
| Storage | `system.storage` | `predictive_optimization_operations_history` |
| Access | `system.access` | `audit`, `table_lineage`, `column_lineage` |

> **Note:** System tables live in the UC `system` catalog and are governed like any other table. A user needs `USE CATALOG` on `system`, `USE SCHEMA` on the schema and `SELECT` on the table, granted by an admin. Each system schema must also be enabled for the metastore. On a new workspace the tables can take a while to populate. Until then, queries may fail with `TABLE_OR_VIEW_NOT_FOUND` or return no rows.

System tables quick reference:

| Table | What It Tracks |
|-------|---------------|
| `system.billing.usage` | DBU consumption by workspace, SKU, cluster, date |
| `system.query.history` | SQL query history: duration, user, warehouse, status |
| `system.compute.clusters` | Cluster events, uptime, compute type |
| `system.access.audit` | Unity Catalog access, permission changes, login events |
| `system.access.table_lineage` | Table-level data lineage (`column_lineage` for columns) |
| `system.lakeflow.job_run_timeline` | Job run history, duration, success/failure |
| `system.lakeflow.pipeline_update_timeline` | Lakeflow pipeline updates: start/end, result state |

> **Note:** Detailed pipeline events (expectation pass/drop counts, flow progress) are in the Beta system table
> `system.lakeflow_pipeline_events_preview.pipeline_events`. For one pipeline, query its event log with the `event_log(TABLE(catalog.schema.table))` TVF.

## Summary

### Security & Governance

| Topic | Key Concept |
|---|---|
| Unity Catalog | 3-level namespace: catalog.schema.table |
| Column Masking | `ALTER TABLE … ALTER COLUMN … SET MASK` |
| Row Filter | `ALTER TABLE … SET ROW FILTER` |
| Access control | `GRANT` / `REVOKE` (UC); `DENY` only in legacy Hive metastore |
| Managed ↔ external | Module 03: `ALTER TABLE … SET MANAGED`; managed → external = recreate |
| Audit Logs *(beyond exam)* | `system.access.audit`: all access events |

## Delta Sharing

> **Self-study:** skipped in class, not required by any lab.

> **Note: beyond the current exam outline, but practical.** Delta Sharing is not listed as an objective in the exam guide updated August 6, 2026. Keep it in your toolbox for real-world data exchange.

Delta Sharing is an open protocol for sharing live Delta Lake data with external recipients. It needs no data copy, and recipients do not need Databricks.

| Concept | Description |
|---------|-------------|
| Share | A named container of schemas/tables/volumes to share |
| Recipient | An external party that receives access. Databricks-to-Databricks: identified by the sharing identifier of its Unity Catalog metastore. Open sharing: authenticates with a bearer token from a credential file (downloaded via an activation link) |
| Provider | The organization sharing data |
| Delta Sharing Server | Databricks Metastore acts as the server |

Key characteristics:
- No data duplication: recipients query live Delta data via HTTP.
- Open protocol: recipients can use pandas, Spark, Power BI or any Delta Sharing client.
- Governed by Unity Catalog: it uses the same GRANT model.

Use it to:
- Share data with external partners without giving them Databricks access.
- Distribute data across clouds or regions.
- Publish datasets to data consumers.

> **Note:** Delta Sharing is part of Unity Catalog governance. It is a _sharing_ mechanism (not a copy) and requires Delta Sharing to be enabled on the metastore.

In [0]:
%sql
-- Delta Sharing: key SQL commands (informational, needs metastore admin or the
-- CREATE SHARE / CREATE RECIPIENT privileges on the metastore)

-- Create a share
-- CREATE SHARE retailhub_share;

-- Add a table to the share (three-part name: your catalog . schema . table, e.g. a gold table)
-- ALTER SHARE retailhub_share ADD TABLE retailhub_<your_name>.gold.gold_customer_orders_summary;

-- Create a recipient
-- Databricks-to-Databricks: identify the recipient's metastore by its sharing identifier
-- CREATE RECIPIENT external_partner USING ID 'azure:<region>:<metastore-uuid>';
-- Open sharing (non-Databricks client): no USING ID
-- -> activation link / bearer-token credential file
-- CREATE RECIPIENT external_partner;

-- Grant the share to the recipient
-- GRANT SELECT ON SHARE retailhub_share TO RECIPIENT external_partner;

-- List this course's shares and recipients. A metastore admin or a user with USE SHARE sees every
-- share (drop LIKE for all of them); anyone else sees only the shares they own. The trainer's
-- screen shows the trainer's course shares; a participant's list is usually empty.
SHOW SHARES LIKE 'retailhub*';
-- SHOW RECIPIENTS;

← <a href="$./09_cicd_and_bundles">09 — CI/CD & Bundles</a> | **<a href="$../../..">Course folder (README)</a>**

## Self-check (trainer): participants can skip this cell

In [0]:
# Self-check (trainer): the ABAC cleanup left nothing behind
# (the cleanup drops the policy and the masked table; the ABAC cells above display their results)
policies = spark.sql(f"SHOW POLICIES ON SCHEMA {CATALOG}.{SILVER_SCHEMA}")
assert policies.filter("`Policy Name` = 'm10_mask_tagged_emails'").count() == 0, "policy still attached to Silver"

print("Self-check passed")